<div dir="rtl" style="text-align:right">
<h1>دو جمع در یک بلوک واقعی</h1><p style="text-align:right"><b>پرسش آزمایش:</b> چرا ورودیِ جمع دوم، نتیجهٔ جمع اول است؟</p><p style="text-align:right">پیش‌نیاز: <a href="http://127.0.0.1:8000/part-06/chapter-02/37-ffn.html"><bdi dir="ltr">37-ffn</bdi></a>، <a href="http://127.0.0.1:8000/part-06/chapter-03/38-residual.html"><bdi dir="ltr">38-residual</bdi></a>، <a href="http://127.0.0.1:8000/part-06/chapter-03/39-layernorm.html"><bdi dir="ltr">39-layernorm</bdi></a>، <a href="http://127.0.0.1:8000/part-06/chapter-04/40-block.html"><bdi dir="ltr">40-block</bdi></a>، <a href="http://127.0.0.1:8000/part-06/chapter-04/41-stack.html"><bdi dir="ltr">41-stack</bdi></a>، <a href="http://127.0.0.1:8000/part-07/chapter-02/45-trace.html"><bdi dir="ltr">45-trace</bdi></a></p><p style="text-align:right">این دفتر مستقل است و به اجرای دفتر دیگری نیاز ندارد. از بالا به پایین اجرا کنید؛ برای تکرار پاک، Kernel را Restart و سپس Run All کنید. لینک درس با سروکردن کتاب روی پورت ۸۰۰۰ کار می‌کند؛ راهنمای نصب در <a href="../../docs/NOTEBOOKS.md"><bdi dir="ltr">docs/NOTEBOOKS.md</bdi></a> است.</p>
</div>

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "mini_gpt" / "model.py").is_file()
             and (p / "data" / "sample.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("Keep notebooks inside the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Python:", sys.executable)
print("Project:", ROOT)

import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(17)

def inspect(name, value):
    print(name, "shape =", tuple(value.shape),
          "dtype =", value.dtype, "device =", value.device)


<div dir="rtl" style="text-align:right">
<p style="text-align:right">بلوک نخستِ MiniGPT واقعی را باز می‌کنیم. معماری این پروژه Pre-Norm است: <code dir="ltr" style="unicode-bidi:isolate">y=x+Attention(LN₁(x))</code> و سپس <code dir="ltr" style="unicode-bidi:isolate">out=y+FFN(LN₂(y))</code>. ترتیب را با Post-Norm جابه‌جا نکنید. وزن‌ها تصادفی و ثابت‌اند.</p>
</div>

In [ ]:
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
config = ModelConfig(vocab_size=12,context_length=8,embedding_dim=12,
                     num_heads=3,num_layers=2,dropout=0.)
model = MiniGPT(config).eval()
ids = torch.tensor([[1,2,3,4,5]],dtype=torch.long)
targets = torch.tensor([[2,3,4,5,6]],dtype=torch.long)
trace = {}
with torch.no_grad():
    logits, loss = model(ids,targets,trace=trace)
    block = model.blocks[0]
    x = trace["combined_embedding"]
    normalized_1 = block.norm_1(x)
    update_1 = block.attention(normalized_1)
    y = x + update_1
    normalized_2 = block.norm_2(y)
    update_2 = block.feed_forward(normalized_2)
    out = y + update_2
    torch.testing.assert_close(update_2,trace["layers"][0]["feed_forward"])
    torch.testing.assert_close(out,trace["layers"][0]["output"])
    hidden = out
    for later in model.blocks[1:]:
        hidden = later(hidden)
    reconstructed = model.language_model_head(model.final_norm(hidden))
    torch.testing.assert_close(reconstructed,logits)
for name,value in [("IDs",ids),("Token Embedding",trace["token_embedding"]),
                   ("Position Embedding",trace["position_embedding"]),("block input",x),
                   ("LN1",normalized_1),("Attention update",update_1),
                   ("first sum",y),("LN2",normalized_2),("FFN update",update_2),
                   ("block output",out),("logits",logits)]:
    inspect(name,value)
print("Loss:",loss.item())


<div dir="rtl" style="text-align:right">
<h2>حذف میان‌بر را دقیق تعریف کنیم</h2><p style="text-align:right">گزینهٔ residual=False هر دو جمع مستقیم را حذف می‌کند؛ LayerNorm و Mask باقی می‌مانند. این کار فقط کم‌کردن x از خروجی نهایی نیست، چون ورودی FFN هم عوض می‌شود. قبل از اجرا Shape و علّیت را پیش‌بینی کنید.</p>
</div>

In [ ]:
with torch.no_grad():
    removed = block(x,residual=False)
    expected = block.feed_forward(block.norm_2(block.attention(block.norm_1(x))))
    torch.testing.assert_close(removed,expected)
    assert removed.shape == out.shape
    changed = ids.clone()
    changed[:,-2:] = torch.tensor([9,10])
    for residual in (True,False):
        original,_ = model(ids,residual=residual)
        modified,_ = model(changed,residual=residual)
        torch.testing.assert_close(original[:,:3],modified[:,:3],rtol=0,atol=1e-7)
print("Shape and causal-prefix checks passed.")
try:
    model(torch.ones(1,config.context_length+1,dtype=torch.long))
except ValueError as error:
    print("Expected context limit:",error)
else:
    raise AssertionError("Expected context-length guard")


<div dir="rtl" style="text-align:right">
<p style="text-align:right"><b>تمرین:</b> یک مدل تازه با context_length=16 بسازید و تعداد پارامترهای جدول موقعیت را مقایسه کنید. سپس فقط C را، با رعایت تقسیم‌پذیری بر H، تغییر دهید. Shapeها و شمار پارامترها را پیش‌بینی کنید؛ انتظار یکسان‌ماندن خروجی عددی مدل تازه نداریم. این آزمون بدون آموزش دربارهٔ کیفیت مدل با یا بدون Residual داوری نمی‌کند.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2>برگشت به کتاب</h2><p style="text-align:right">پیش‌بینی، مشاهده و دلیل اختلافشان را در یادداشت خود بنویسید. سپس به <a href="http://127.0.0.1:8000/part-07/chapter-02/45-trace.html">درس مرتبط</a> برگردید و نتیجه را با توضیح آن مقایسه کنید.</p>
</div>